In [0]:
from datetime import datetime

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.table_read_log (
    table_id INT,
    message STRING,
    run_time TIMESTAMP
)
""")

print("Logging table is ready")

In [0]:
dbutils.widgets.text("table_id", "1")

table_id = int(dbutils.widgets.get("table_id"))

print(f"Received table ID: {table_id}")

In [0]:
class TableReader:

    def __init__(self, control_table="workspace.default.tableId"):
        self.control_table = control_table

    def read_table(self, table_id):

        row = (
            spark.table(self.control_table)
            .filter(f"table_id = {table_id}")
            .first()
        )

        if row is None:
            raise ValueError(
                f"table_id {table_id} not found"
            )

        return f"table read successfully {table_id}"

In [0]:
reader = TableReader()

result = reader.read_table(table_id)

print(result)

# Create execution log
log_df = spark.createDataFrame(
    [(table_id, result, datetime.now())],
    """
    table_id INT,
    message STRING,
    run_time TIMESTAMP
    """
)

# Insert into logging table
log_df.write \
    .mode("append") \
    .saveAsTable("workspace.default.table_read_log")

print("Execution log inserted successfully")

In [0]:
dbutils.notebook.exit(result)